# 1. Data and task

Training cohort: **TCGA-BRCA** (RNA-seq). Validation cohort: **METABRIC** (Illumina microarray).
This notebook checks the files, builds the labels, and shows why TP53 mutation status is the task.

Run it from the project folder, so that `pqk100/`, `data/` and `results/` are next to it.

In [1]:
import os, sys
from pathlib import Path
# Find the project folder (the one holding the pqk100 package), wherever Jupyter or VS Code started
def _find_root():
    starts = [Path(p).parent for p in [globals().get("__vsc_ipynb_file__")] if p]
    starts += [Path.cwd(), Path(r"D:\Active Research Projects\PQK100")]
    for s in starts:
        for p in [s, *s.parents]:
            if (p / "pqk100" / "__init__.py").exists() and (p / "credits").is_dir():
                return p.resolve()
    raise RuntimeError(f"Cannot find the PQK100 project folder from {Path.cwd()}. "
                       "Set ROOT = Path(r'D:\\Active Research Projects\\PQK100') by hand in this cell.")
ROOT = _find_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("project folder:", ROOT)

import os, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

from pqk100.data import load_cohorts
from pqk100 import study

project folder: D:\Active Research Projects\PQK100


## 1.1 Files

Nine files from the cBioPortal datahub, about 880 MB. If any is missing, run the next cell.
It skips files that are already there and resumes an interrupted download.

In [2]:
FILES = ["data/tcga_brca/expr.txt", "data/tcga_brca/clin_patient.txt", "data/tcga_brca/clin_sample_pub2015.txt",
         "data/tcga_brca/mutations_pub2015.txt", "data/tcga_brca/cases_sequenced_pub2015.txt",
         "data/metabric/expr.txt", "data/metabric/clin_patient.txt", "data/metabric/mutations.txt",
         "data/metabric/cases_sequenced.txt"]
have_cache = os.path.exists("cache/cohorts.npz")
for f in FILES:
    print(f"{'ok     ' if os.path.exists(f) else 'MISSING'}  {f}" + (f"  ({os.path.getsize(f)/1e6:.1f} MB)" if os.path.exists(f) else ""))
if any(f.endswith(".part") for d in ("data/tcga_brca", "data/metabric") if os.path.isdir(d) for f in os.listdir(d)):
    print("\nA download is still in progress (.part file present).")

ok       data/tcga_brca/expr.txt  (154.7 MB)
ok       data/tcga_brca/clin_patient.txt  (0.3 MB)
ok       data/tcga_brca/clin_sample_pub2015.txt  (0.8 MB)
ok       data/tcga_brca/mutations_pub2015.txt  (34.6 MB)
ok       data/tcga_brca/cases_sequenced_pub2015.txt  (0.0 MB)
ok       data/metabric/expr.txt  (689.3 MB)
ok       data/metabric/clin_patient.txt  (0.4 MB)
ok       data/metabric/mutations.txt  (4.0 MB)
ok       data/metabric/cases_sequenced.txt  (0.0 MB)


In [3]:
# The data are downloaded by 00_setup_and_data.ipynb.

## 1.2 Load both cohorts

The first call parses the text files and writes `cache/cohorts.npz`. That takes about a minute.
Later calls read the cache. Expression is on a log2 scale and restricted to genes present in both cohorts.

In [4]:
tcga, meta = load_cohorts()
rows = []
for c in (tcga, meta):
    for task in c.labels:
        y = c.labels[task].dropna()
        rows.append({"cohort": c.name, "tumors": len(c.X), "genes": c.X.shape[1], "task": task,
                     "labeled": len(y), "positive fraction": round(y.mean(), 3)})
pd.DataFrame(rows).set_index(["cohort", "task"])

tumors  genes  labeled  positive fraction
cohort    task                                            
TCGA-BRCA lumAB    1082  17139      696              0.283
          ER       1082  17139      771              0.773
          TP53     1082  17139      811              0.344
METABRIC  lumAB    1980  17139     1175              0.404
          ER       1980  17139     1937              0.773
          TP53     1980  17139     1904              0.346

Labels:

- `TP53`: non-silent somatic TP53 mutation, from DNA sequencing.
- `ER`: estrogen receptor status by immunohistochemistry.
- `lumAB`: PAM50 luminal A (0) against luminal B (1).

## 1.3 Which task?

Classical models only. Train on 200 TCGA tumors, score on held-out TCGA and on METABRIC.
A good task is not solved by one gene, leaves headroom, and drops on the independent cohort.

`RERUN = True` recomputes (about 5 minutes). Otherwise the saved table is read.

In [5]:
RERUN = False
path = "results/task_screen.csv"
screen = study.task_screen(out=path) if (RERUN or not os.path.exists(path)) else pd.read_csv(path)
t = screen[screen.scaling == "cohort"].groupby(["task", "k", "model"])[["in_auc", "ext_auc"]].mean().round(3)
t.columns = ["AUC held-out TCGA", "AUC METABRIC"]
t

AUC held-out TCGA  AUC METABRIC
task  k   model                                   
ER    1   logreg               0.957         0.957
      20  gbm                  0.946         0.918
          logreg               0.957         0.938
          rbf_svm              0.957         0.931
      156 gbm                  0.938         0.927
          logreg               0.951         0.941
          rbf_svm              0.952         0.935
TP53  1   logreg               0.864         0.791
      20  gbm                  0.897         0.856
          logreg               0.921         0.875
          rbf_svm              0.910         0.864
      156 gbm                  0.917         0.842
          logreg               0.925         0.874
          rbf_svm              0.921         0.878
lumAB 1   logreg               0.882         0.840
      20  gbm                  0.943         0.917
          logreg               0.950         0.931
          rbf_svm              0.942         0.915
      156 gbm                  0.956         0.926
          logreg               0.973         0.929
          rbf_svm              0.969         0.926

In [6]:
print("Best single gene per task (count over resamples and normalizations):")
print(screen.groupby("task").top_gene.agg(lambda s: s.value_counts().to_dict()).to_string())

Best single gene per task (count over resamples and normalizations):
task
ER                                            {'ESR1': 42}
TP53               {'PSAT1': 21, 'TBC1D9': 14, 'A2ML1': 7}
lumAB    {'SPAG5': 14, 'CEP55': 14, 'AURKA': 7, 'EXO1': 7}


Reading the table:

- **ER** is solved by ESR1 alone. More genes add nothing.
- **Luminal A vs B** is a label computed from expression, and classical models already reach about 0.97.
- **TP53** needs many genes, stays near 0.92, and drops to about 0.87 on METABRIC. The label is independent of the expression data.

TP53 is the task used in the rest of the project.

## 1.4 Normalization across platforms

RNA-seq and microarray values are on different scales. Two options are implemented in `pqk100/prep.py`:

- `cohort`: z-score each gene inside its own cohort. Uses no labels, but does use the unlabeled validation data.
- `rank`: rank-transform each sample on its own, then scale with training statistics.

The comparison below is for TP53 with 156 genes.

In [7]:
t = screen[(screen.task == "TP53") & (screen.k == 156)].groupby(["scaling", "model"])[["in_auc", "ext_auc"]].mean().round(3)
t.columns = ["AUC held-out TCGA", "AUC METABRIC"]
t

AUC held-out TCGA  AUC METABRIC
scaling model                                   
cohort  gbm                  0.917         0.842
        logreg               0.925         0.874
        rbf_svm              0.921         0.878
rank    gbm                  0.924         0.858
        logreg               0.925         0.865
        rbf_svm              0.925         0.848